# Model Experiment — `TFT`

## Theory

**Temporal Fusion Transformer (TFT)** is an attention-based deep model built
specifically to fuse three kinds of inputs per series: **static** covariates
(things that don't change over time -- here Store `Type`/`Size`), **known
future** covariates (things known in advance for the forecast horizon -- here
calendar/holiday flags and, since `features.csv` already covers the whole test
period, Temperature/Fuel_Price/CPI/Unemployment/MarkDowns too), and **past
observed** covariates/target history. It has built-in *Variable Selection
Networks* that learn which inputs matter per time step, plus interpretable
multi-head attention over the lookback window.

**Contrast with N-BEATS (this repo's other DL model):** N-BEATS is purely
univariate by design -- no exogenous or static inputs at all. TFT is the
opposite end of the same DL family: it's built to exploit exactly the tabular
features (`features.py`-style calendar/markdown/store attributes) that the
tree models also use, but through attention instead of splits. Comparing the
two isolates how much the holiday/markdown/store-size signal is actually worth
to a neural forecaster, independent of architecture family (tree vs DL).

**Global + direct multi-horizon:** like N-BEATS, one TFT model is trained
across all ~3,000 series at once (weights shared, static covariates tell it
which series is which), and it forecasts all `h` steps directly in one pass,
so `h` is fixed at construction time (CV uses `h=8`, the Final run uses `h` =
the full Kaggle test horizon).

**Tracking:** Weights & Biases (assignment allows this for neural network
architectures).

## 0. Config & environment (Kaggle vs local)

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_COMPETITION = 'walmart-recruiting-store-sales-forecasting'
ON_KAGGLE = KAGGLE_INPUT.exists()

if ON_KAGGLE:
    RAW_DIR = KAGGLE_INPUT / KAGGLE_COMPETITION
    WORKING_DIR = Path('/kaggle/working')
else:
    PROJECT_ROOT = Path.cwd().parent
    RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
    WORKING_DIR = PROJECT_ROOT
    load_dotenv(PROJECT_ROOT / '.env')

def _resolve(stem):
    for name in (f'{stem}.csv', f'{stem}.csv.zip'):
        p = RAW_DIR / name
        if p.exists():
            return p
    return RAW_DIR / f'{stem}.csv'

TRAIN_CSV = _resolve('train')
TEST_CSV = _resolve('test')
FEATURES_CSV = _resolve('features')
STORES_CSV = _resolve('stores')

RANDOM_SEED = 42
TARGET = 'Weekly_Sales'
HOLIDAY_WEIGHT = 5
NON_HOLIDAY_WEIGHT = 1

if ON_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for key in ('MLFLOW_TRACKING_URI', 'MLFLOW_TRACKING_USERNAME', 'MLFLOW_TRACKING_PASSWORD'):
            try:
                os.environ.setdefault(key, client.get_secret(key))
            except Exception:
                pass
    except Exception:
        pass

MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI')
MLFLOW_TRACKING_USERNAME = os.getenv('MLFLOW_TRACKING_USERNAME')
MLFLOW_TRACKING_PASSWORD = os.getenv('MLFLOW_TRACKING_PASSWORD')

print('On Kaggle:', ON_KAGGLE, '| raw data dir:', RAW_DIR)

## 1. Data loading helpers

In [ ]:
import pandas as pd

def _read_bool(series):
    if series.dtype == bool:
        return series
    return series.astype(str).str.strip().str.upper().map({'TRUE': True, 'FALSE': False})

def load_stores():
    return pd.read_csv(STORES_CSV)

def load_features():
    df = pd.read_csv(FEATURES_CSV, parse_dates=['Date'])
    df['IsHoliday'] = _read_bool(df['IsHoliday'])
    df = df.sort_values(['Store', 'Date'])
    for col in ('CPI', 'Unemployment'):
        df[col] = df.groupby('Store')[col].transform(lambda s: s.ffill().bfill())
    return df.reset_index(drop=True)

def load_raw(split):
    path = TRAIN_CSV if split == 'train' else TEST_CSV
    df = pd.read_csv(path, parse_dates=['Date'])
    df['IsHoliday'] = _read_bool(df['IsHoliday'])
    return df

def load_merged(split='train'):
    base = load_raw(split)
    stores = load_stores()
    feats = load_features().drop(columns=['IsHoliday'])
    df = base.merge(stores, on='Store', how='left')
    df = df.merge(feats, on=['Store', 'Date'], how='left')
    df = df.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
    return df

## 2. Metric (WMAE) & cross-validation helpers

In [ ]:
def weights_from_holiday(is_holiday):
    is_holiday = np.asarray(is_holiday).astype(bool)
    return np.where(is_holiday, HOLIDAY_WEIGHT, NON_HOLIDAY_WEIGHT).astype(float)

def wmae(y_true, y_pred, is_holiday):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    w = weights_from_holiday(is_holiday)
    return float(np.sum(w * np.abs(y_true - y_pred)) / np.sum(w))

In [ ]:
def _sorted_unique_dates(dates):
    return np.sort(pd.to_datetime(dates).unique())

def time_holdout(df, weeks=8, date_col='Date'):
    uniq = _sorted_unique_dates(df[date_col])
    if weeks >= len(uniq):
        raise ValueError(f'weeks={weeks} >= number of distinct weeks {len(uniq)}')
    cutoff = uniq[-weeks]
    d = pd.to_datetime(df[date_col]).to_numpy()
    train_idx = np.where(d < cutoff)[0]
    val_idx = np.where(d >= cutoff)[0]
    return train_idx, val_idx

def expanding_splits(df, n_splits=3, horizon=8, date_col='Date'):
    uniq = _sorted_unique_dates(df[date_col])
    needed = horizon * n_splits
    if needed >= len(uniq):
        raise ValueError(f'Need > {needed} distinct weeks for {n_splits} folds of {horizon}; have {len(uniq)}.')
    d = pd.to_datetime(df[date_col]).to_numpy()
    for k in range(n_splits):
        end_offset = needed - k * horizon
        start_offset = end_offset - horizon
        val_start = uniq[-end_offset]
        val_end = uniq[-start_offset] if start_offset > 0 else None
        train_idx = np.where(d < val_start)[0]
        if val_end is None:
            val_idx = np.where(d >= val_start)[0]
        else:
            val_idx = np.where((d >= val_start) & (d < val_end))[0]
        yield train_idx, val_idx

## 3. Feature engineering helpers (for future-known/static exogenous inputs)

In [ ]:
import numpy as np

MARKDOWN_COLS = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']

def add_calendar_features(df):
    df = df.copy()
    d = df['Date'].dt
    df['Year'] = d.year
    df['Month'] = d.month
    df['Week'] = d.isocalendar().week.astype(int)
    df['Day'] = d.day
    df['DayOfYear'] = d.dayofyear
    df['Week_sin'] = np.sin(2 * np.pi * df['Week'] / 52)
    df['Week_cos'] = np.cos(2 * np.pi * df['Week'] / 52)
    df['Month_sin'] = np.sin(2 * np.pi * df['Month'] / 12)
    df['Month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)
    df['IsSuperBowl'] = df['Week'].isin([6]).astype(int)
    df['IsLaborDay'] = df['Week'].isin([36]).astype(int)
    df['IsThanksgiving'] = df['Week'].isin([47]).astype(int)
    df['IsChristmas'] = df['Week'].isin([51, 52]).astype(int)
    return df

def clean_markdowns(df):
    df = df.copy()
    for col in MARKDOWN_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    df['MarkDown_total'] = df[MARKDOWN_COLS].sum(axis=1)
    df['MarkDown_active'] = (df['MarkDown_total'] > 0).astype(int)
    return df

def encode_store_type(df):
    df = df.copy()
    if 'Type' in df.columns:
        df['Type_code'] = df['Type'].map({'A': 0, 'B': 1, 'C': 2}).astype('Int64')
    return df

def add_lag_features(df, target='Weekly_Sales', lags=(1, 2, 3, 4, 52), rolling_windows=(4, 8, 12)):
    df = df.sort_values(['Store', 'Dept', 'Date']).copy()
    g = df.groupby(['Store', 'Dept'])[target]
    for lag in lags:
        df[f'{target}_lag{lag}'] = g.shift(lag)
    for w in rolling_windows:
        df[f'{target}_rollmean{w}'] = g.transform(lambda s: s.shift(1).rolling(w).mean())
        df[f'{target}_rollstd{w}'] = g.transform(lambda s: s.shift(1).rolling(w).std())
    return df

def build_features(df, add_lags=False):
    out = add_calendar_features(df)
    out = clean_markdowns(out)
    out = encode_store_type(out)
    if add_lags and 'Weekly_Sales' in out.columns:
        out = add_lag_features(out)
    return out

NON_FEATURE_COLS = ['Date', 'Type', 'Weekly_Sales']

def feature_columns(df):
    cols = [c for c in df.columns if c not in NON_FEATURE_COLS]
    return [c for c in cols if pd.api.types.is_numeric_dtype(df[c])]

## 4. Feature selection helpers

In [ ]:
from sklearn.ensemble import RandomForestRegressor

def drop_low_variance(df, feature_cols, threshold=1e-6):
    variances = df[feature_cols].var(numeric_only=True)
    keep = [c for c in feature_cols if variances.get(c, 0.0) > threshold]
    dropped = [c for c in feature_cols if c not in keep]
    return keep, dropped

def drop_correlated(df, feature_cols, threshold=0.95):
    corr = df[feature_cols].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > threshold)]
    keep = [c for c in feature_cols if c not in to_drop]
    return keep, to_drop

def rank_by_importance(df, feature_cols, target='Weekly_Sales', sample_size=60_000, random_state=42):
    data = df[feature_cols + [target]].dropna(subset=[target]).fillna(0)
    if len(data) > sample_size:
        data = data.sample(sample_size, random_state=random_state)
    rf = RandomForestRegressor(n_estimators=100, max_depth=10, n_jobs=-1, random_state=random_state)
    rf.fit(data[feature_cols], data[target])
    return pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

def select_features(df, feature_cols, target='Weekly_Sales', top_k=20, variance_threshold=1e-6, corr_threshold=0.95, random_state=42):
    after_var, dropped_var = drop_low_variance(df, feature_cols, variance_threshold)
    after_corr, dropped_corr = drop_correlated(df, after_var, corr_threshold)
    importances = rank_by_importance(df, after_corr, target=target, random_state=random_state)
    selected = list(importances.index) if top_k is None else list(importances.index[:top_k])
    return {
        'n_input_features': len(feature_cols),
        'dropped_low_variance': dropped_var,
        'dropped_correlated': dropped_corr,
        'importances': importances.to_dict(),
        'selected_features': selected,
        'n_selected_features': len(selected),
    }

## 5. Global-model (neuralforecast) shared helpers

In [ ]:
from sklearn.base import BaseEstimator, RegressorMixin

NF_FREQ = 'W-FRI'

def make_unique_id(df):
    return df['Store'].astype(str) + '_' + df['Dept'].astype(str)

def cold_start_fallback_table(long_train):
    """Per-unique_id recent mean, plus a global mean for series never seen in training."""
    per_id = long_train.groupby('unique_id')['y'].apply(lambda s: s.tail(8).mean())
    global_mean = float(long_train['y'].mean())
    return per_id, global_mean

def fill_cold_start(pred_df, value_col, fallback_per_id, global_mean):
    missing = pred_df[value_col].isna()
    if missing.any():
        pred_df.loc[missing, value_col] = pred_df.loc[missing, 'unique_id'].map(fallback_per_id)
        pred_df[value_col] = pred_df[value_col].fillna(global_mean)
    return pred_df

## 6. Weights & Biases setup

In [ ]:
import wandb
import numpy as np, pandas as pd

MODEL_NAME = 'TFT'
WANDB_PROJECT = 'walmart-sales-forecasting'
wandb.login()

## Load raw data

In [ ]:
raw_train = load_raw('train')
raw_test  = load_raw('test')
raw_train.shape, raw_test.shape

## Run 1 — Cleaning
Same continuous-weekly-panel construction as N-BEATS, but we also need to keep
the merged exogenous columns (calendar/markdown/store) aligned to that panel
for the Feature Selection + Pipeline steps below.

In [ ]:
def merge_exog(df):
    stores = load_stores()
    feats = load_features().drop(columns=['IsHoliday'])
    out = df.merge(stores, on='Store', how='left').merge(feats, on=['Store', 'Date'], how='left')
    return build_features(out, add_lags=False)

merged_train = merge_exog(raw_train)
n_series = merged_train.groupby(['Store', 'Dept']).ngroups
lengths = merged_train.groupby(['Store', 'Dept']).size()
span_weeks = merged_train.groupby(['Store', 'Dept'])['Date'].agg(lambda d: (d.max() - d.min()).days // 7 + 1)
n_with_gaps = int((lengths < span_weeks).sum())

run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type='cleaning', name=f'{MODEL_NAME}_Cleaning')
wandb.log({
    'n_series': n_series,
    'n_series_with_gaps': n_with_gaps,
    'n_train_rows': len(raw_train),
    'n_negative_sales': int((raw_train.Weekly_Sales < 0).sum()),
})
run.finish()
print(n_series, 'series,', n_with_gaps, 'with gaps')

## Run 2 — Feature Selection
Build the same engineered tabular features used by the tree models, then run
the shared selection pipeline (drop zero-variance/redundant columns, rank the
rest by RandomForest importance) to choose which engineered columns become
TFT's `futr_exog_list`. Store `Type_code`/`Size` are used directly as
`stat_exog_list` -- they're the only truly time-invariant attributes available,
so there's nothing to select there.

In [ ]:
candidate_cols = [c for c in feature_columns(merged_train) if c not in ('Store', 'Dept', 'Type_code', 'Size')]
fs_result = select_features(merged_train, candidate_cols, target='Weekly_Sales', top_k=10)
futr_exog_list = fs_result['selected_features']
stat_exog_list = ['Type_code', 'Size']

run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type='feature_selection', name=f'{MODEL_NAME}_Feature_Selection')
wandb.config.update({
    'n_input_features': fs_result['n_input_features'],
    'dropped_low_variance': fs_result['dropped_low_variance'],
    'dropped_correlated': fs_result['dropped_correlated'],
    'futr_exog_list': futr_exog_list,
    'stat_exog_list': stat_exog_list,
})
wandb.log({f'importance_{k}': v for k, v in fs_result['importances'].items()})
run.finish()
print('futr_exog_list:', futr_exog_list)
print('stat_exog_list:', stat_exog_list)

## 7. Pipeline wrapper (raw-test-ready, wraps neuralforecast TFT)

In [ ]:
from neuralforecast import NeuralForecast
from neuralforecast.models import TFT
from neuralforecast.losses.pytorch import MAE

class TFTPipeline(BaseEstimator, RegressorMixin):
    """sklearn-style wrapper: fit(raw_train)/predict(raw_test), same contract as
    every other model in this repo. Internally merges stores/features (like
    WalmartPreprocessor did) to build futr_exog/stat_exog inputs for TFT."""

    def __init__(self, horizon=39, input_size=52, hidden_size=32, n_head=4,
                 learning_rate=1e-3, max_steps=300,
                 futr_exog_list=None, stat_exog_list=None, random_seed=42):
        self.horizon = horizon
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.n_head = n_head
        self.learning_rate = learning_rate
        self.max_steps = max_steps
        self.futr_exog_list = futr_exog_list or []
        self.stat_exog_list = stat_exog_list or []
        self.random_seed = random_seed

    def _to_long(self, df, with_target):
        merged = merge_exog(df)
        cols = ['unique_id', 'ds'] + self.futr_exog_list
        long = pd.DataFrame({'unique_id': make_unique_id(merged), 'ds': pd.to_datetime(merged['Date'])})
        for c in self.futr_exog_list:
            long[c] = merged[c].to_numpy()
        if with_target:
            long['y'] = merged['Weekly_Sales'].to_numpy(dtype=float)
        return long, merged

    def _static_df(self, merged):
        static = merged[['Store', 'Dept'] + self.stat_exog_list].drop_duplicates(['Store', 'Dept'])
        static['unique_id'] = make_unique_id(static)
        return static[['unique_id'] + self.stat_exog_list].reset_index(drop=True)

    def fit(self, X, y=None):
        df = X.copy()
        df['Weekly_Sales'] = np.asarray(y, dtype=float)
        long, merged = self._to_long(df, with_target=True)
        self.long_train_ = long
        self.static_df_ = self._static_df(merged)
        self.fallback_per_id_, self.global_mean_ = cold_start_fallback_table(long)

        model = TFT(
            h=self.horizon, input_size=self.input_size, hidden_size=self.hidden_size,
            n_head=self.n_head, learning_rate=self.learning_rate, max_steps=self.max_steps,
            futr_exog_list=self.futr_exog_list, stat_exog_list=self.stat_exog_list,
            loss=MAE(), random_seed=self.random_seed,
        )
        self.nf_ = NeuralForecast(models=[model], freq=NF_FREQ)
        self.nf_.fit(df=self.long_train_, static_df=self.static_df_)
        self.model_col_ = 'TFT'
        return self

    def predict(self, X):
        long, _ = self._to_long(X, with_target=False)
        fcst = self.nf_.predict(futr_df=long[['unique_id', 'ds'] + self.futr_exog_list])
        merged_pred = long[['unique_id', 'ds']].merge(fcst[['unique_id', 'ds', self.model_col_]], on=['unique_id', 'ds'], how='left')
        merged_pred = fill_cold_start(merged_pred, self.model_col_, self.fallback_per_id_, self.global_mean_)
        return merged_pred[self.model_col_].to_numpy()

## Run 3 — Cross-validation (curated hyperparameter search)
Random search over 10 configs (`hidden_size`, `n_head`, `learning_rate`,
`input_size`), each evaluated with an 8-week expanding-window backtest. One
Wandb run per trial. `max_steps` is kept modest for search time; raise it (and
`hidden_size`) for the Final run on a GPU.

In [ ]:
import random

search_space = {
    'hidden_size': [16, 32, 64],
    'n_head': [2, 4],
    'learning_rate': [1e-3, 5e-4, 1e-4],
    'input_size': [26, 52],
}
rng = random.Random(RANDOM_SEED)
keys = list(search_space)
N_TRIALS = 10
configs = []
seen = set()
while len(configs) < N_TRIALS:
    cfg = {k: rng.choice(v) for k, v in search_space.items()}
    key = tuple(sorted(cfg.items()))
    if key not in seen:
        seen.add(key)
        configs.append(cfg)
print(len(configs), 'configs to try')

trial_results = []
for i, cfg in enumerate(configs):
    run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type='cv', name=f'{MODEL_NAME}_CV_trial{i}', config=cfg, reinit=True)
    fold_scores = []
    for k, (tr_idx, va_idx) in enumerate(expanding_splits(raw_train, n_splits=3, horizon=8)):
        tr, va = raw_train.iloc[tr_idx], raw_train.iloc[va_idx]
        pipe = TFTPipeline(horizon=8, max_steps=150, futr_exog_list=futr_exog_list, stat_exog_list=stat_exog_list, **cfg)
        pipe.fit(tr, tr['Weekly_Sales'])
        pred = pipe.predict(va)
        score = wmae(va['Weekly_Sales'], pred, va['IsHoliday'])
        fold_scores.append(score)
        wandb.log({'fold': k, 'fold_wmae': score})
    cv_mean = float(np.mean(fold_scores))
    wandb.log({'cv_wmae_mean': cv_mean, 'cv_wmae_std': float(np.std(fold_scores))})
    run.finish()
    trial_results.append({'config': cfg, 'cv_wmae_mean': cv_mean})
    print(f'config {i} {cfg}: CV WMAE={cv_mean:,.1f}')

best_trial = min(trial_results, key=lambda t: t['cv_wmae_mean'])
best_config = best_trial['config']
print('best config:', best_config, '| CV WMAE:', best_trial['cv_wmae_mean'])

## Run 4 — Final fit + save Pipeline
Refit the chosen config on the **full** raw_train with `horizon` = the number
of weeks in the Kaggle test period, check holdout WMAE, and save the fitted
Pipeline. Same note as N-BEATS: Wandb has no MLflow-style Model Registry, so if
TFT ends up the overall best architecture, wrap it as an `mlflow.pyfunc` model
at that point (see note below).

In [ ]:
TEST_HORIZON = raw_test['Date'].nunique()
print('forecast horizon:', TEST_HORIZON, 'weeks')

run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type='final', name=f'{MODEL_NAME}_Final', config=best_config)

holdout_tr, holdout_va = time_holdout(raw_train, weeks=8)
p = TFTPipeline(horizon=8, max_steps=300, futr_exog_list=futr_exog_list, stat_exog_list=stat_exog_list, **best_config)
p.fit(raw_train.iloc[holdout_tr], raw_train.iloc[holdout_tr]['Weekly_Sales'])
hv = raw_train.iloc[holdout_va]
holdout_wmae = wmae(hv['Weekly_Sales'], p.predict(hv), hv['IsHoliday'])
wandb.log({'holdout_wmae': holdout_wmae})
print('holdout WMAE:', holdout_wmae)

final_pipe = TFTPipeline(horizon=TEST_HORIZON, max_steps=300, futr_exog_list=futr_exog_list, stat_exog_list=stat_exog_list, **best_config)
final_pipe.fit(raw_train, raw_train['Weekly_Sales'])

import pickle, pathlib
out_dir = pathlib.Path(WORKING_DIR) / 'models' / MODEL_NAME
out_dir.mkdir(parents=True, exist_ok=True)
final_pipe.nf_.save(path=str(out_dir / 'nf_model'), overwrite=True)
with open(out_dir / 'pipeline_wrapper.pkl', 'wb') as f:
    pickle.dump({
        'horizon': TEST_HORIZON, 'futr_exog_list': futr_exog_list,
        'stat_exog_list': stat_exog_list, **best_config,
    }, f)
wandb.log_artifact(str(out_dir), name=f'{MODEL_NAME}_pipeline', type='model')
run.finish()
print('saved pipeline to', out_dir)

> **If TFT turns out to be the overall best model:** wrap `final_pipe` (or
> reload via `NeuralForecast.load(path=...)` + the pickled wrapper config) in
> an `mlflow.pyfunc.PythonModel` and log/register that with
> `mlflow.pyfunc.log_model(...)` + `mlflow.register_model(...)` so
> `model_inference.ipynb` can load it from the Model Registry the same way as
> the sklearn-based Pipelines.